In [1]:
# Install LIbraries
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 677.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title Simple Interactive Query with DPO-aligned Model (Hugging Face)

from unsloth import FastLanguageModel
from huggingface_hub import login
from google.colab import userdata
import torch
import sys

# Helper to clear GPU memory (defined in GXiIVmyvcssh)
def clear_gpu_memory():
    import gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()

# Retrieve Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face Hub
if hf_token:
    try:
        login(token=hf_token)
        print("Successfully logged in to Hugging Face.")
    except Exception as e:
        print(f"Hugging Face login failed: {e}. Model loading might fail if repo is private.")
else:
    print("HF_TOKEN not found in Colab secrets. Model loading might fail if private repo.")

# Define the Hugging Face repository name where the model was pushed
hf_repo_name = "mnayak1/my-it-helpdesk-dpo-model"

# Clear GPU memory before loading the model to ensure enough resources
clear_gpu_memory()

# Load the DPO-aligned model and tokenizer for interactive inference
print(f"Loading DPO-aligned model from Hugging Face: {hf_repo_name}...")

try:
    # Ensure MAX_SEQ_LENGTH is defined, use a default if not found
    if 'MAX_SEQ_LENGTH' not in globals(): # Use globals() instead of locals() for global variables
        MAX_SEQ_LENGTH = 512

    interactive_hf_model, interactive_hf_tokenizer = FastLanguageModel.from_pretrained(
        model_name=hf_repo_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if interactive_hf_tokenizer.pad_token is None:
        interactive_hf_tokenizer.pad_token = interactive_hf_tokenizer.eos_token
    interactive_hf_tokenizer.padding_side = "right"

    print("DPO-aligned model loaded successfully from Hugging Face.")
except Exception as e:
    print(f"Error loading DPO model from Hugging Face: {e}")
    interactive_hf_model = None
    interactive_hf_tokenizer = None

# Check if the model loaded successfully
if interactive_hf_model is None or interactive_hf_tokenizer is None:
    print("Model not available for inference. Exiting interactive session.")
else:
    print("\n--- Interactive IT Helpdesk Assistant (Hugging Face Model) ---")
    print("Type your IT helpdesk question and press Enter. Type 'exit' to quit.")

    while True:
        try:
            question = input("\nYour Question: ")
            if question.lower() == 'exit':
                print("Exiting interactive session.")
                break

            if not question.strip():
                print("Please enter a question.")
                continue

            # Use the generate_answer function (assuming it's defined in another cell, GXiIVmyvcssh)
            # If generate_answer is not yet defined, it will raise a NameError here.
            # The original agent context states GXiIVmyvcssh defines it, so this should be fine.
            print(f"Generating answer for: {question}")
            try:
                generated_answer = generate_answer(
                    model=interactive_hf_model,
                    tokenizer=interactive_hf_tokenizer,
                    instruction=question.strip(),
                    max_new_tokens=256
                )
                print("\n--- DPO Model Answer (from Hugging Face) ---")
                print(generated_answer)
            except NameError:
                print("Error: 'generate_answer' function not found. Please ensure the cell defining helper functions (GXiIVmyvcssh) has been run.")
            except Exception as e:
                print(f"An error occurred during generation: {e}")

        except EOFError:
            print("Exiting interactive session due to EOF.")
            break
        except KeyboardInterrupt:
            print("Exiting interactive session due to keyboard interrupt.")
            break

# Clean up GPU memory after the interactive session
if interactive_hf_model is not None:
    del interactive_hf_model
if interactive_hf_tokenizer is not None:
    del interactive_hf_tokenizer
clear_gpu_memory()